# CLAM-TR Thesis Figures - Step 1 Colab Notebook

**Generated 2026-04-29.** Reads existing v20 output JSONs in
`MyDrive/CLAM_TR_Results/v20_fullscale/{UNI, UNI2-h}/` and produces 7 thesis
figures plus a BH-FDR + Bonferroni cross-check writeback to `statistical_tests.json`.

**No GPU needed** - CPU runtime is fine. Total runtime ~5 min.

**Run order:** top to bottom. Cell 0 mounts Drive. Cells 1-7 each save a PDF.
Cell 8 verifies all 7 PDFs landed.

**Output folder:** `MyDrive/thesis/figures/` (created automatically by Cell 0).

**Closes SUGGESTIONS:** S013, S022, S023, S024, S025, S027, S028, S029 (8 of 27 thesis assets).

## CELL 0 — Setup (mount Drive + load JSONs)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import confusion_matrix
from statsmodels.stats.multitest import fdrcorrection, multipletests

DRIVE = Path("/content/drive/MyDrive")
RESULTS = DRIVE / "CLAM_TR_Results" / "v20_fullscale"
UNI_DIR = RESULTS / "UNI"
UNI2H_DIR = RESULTS / "UNI2-h"
FIG_DIR = DRIVE / "thesis" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

FLOOR_UNI, FLOOR_UNI2H = 0.8121, 0.8414
VARIANTS = ["ABMIL", "DualBranch", "RetNet", "SpatialMult", "SpatialMultRetNet", "SpatialAdd"]

with open(UNI_DIR / "v20_final_summary.json") as f:
    uni_summary = json.load(f)
with open(UNI2H_DIR / "v20_final_summary.json") as f:
    uni2h_summary = json.load(f)

uni_exp   = uni_summary["experiments"]
uni2h_exp = uni2h_summary["experiments"]

sns.set_style("whitegrid")
plt.rcParams.update({
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "legend.fontsize": 9, "figure.dpi": 100, "savefig.dpi": 300,
    "savefig.bbox": "tight", "pdf.fonttype": 42,
})
print(f"Setup OK. UNI experiments: {len(uni_exp)} | UNI2-h experiments: {len(uni2h_exp)}")
print(f"Sample variant keys (UNI): {list(uni_exp.keys())[:6]}")

## CELL 1 — S022 QWK comparison bar chart

In [ ]:
def get_bestHP(experiments_dict, variant):
    rec = experiments_dict.get(f"{variant}_bestHP", {})
    return rec.get("mean_qwk", np.nan), rec.get("std_qwk", np.nan)

uni_means = [get_bestHP(uni_exp, v)[0] for v in VARIANTS]
uni_stds  = [get_bestHP(uni_exp, v)[1] for v in VARIANTS]
uni2h_means = [get_bestHP(uni2h_exp, v)[0] for v in VARIANTS]
uni2h_stds  = [get_bestHP(uni2h_exp, v)[1] for v in VARIANTS]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(VARIANTS)); w = 0.38
ax.bar(x - w/2, uni_means, w, yerr=uni_stds, capsize=3, label="UNI (ViT-L/16)", color="#2b6cb0", edgecolor="black")
ax.bar(x + w/2, uni2h_means, w, yerr=uni2h_stds, capsize=3, label="UNI2-h (ViT-H/14)", color="#dd6b20", edgecolor="black")
ax.axhline(FLOOR_UNI,   ls="--", lw=1, color="#2b6cb0", alpha=0.6, label=f"UNI floor ({FLOOR_UNI})")
ax.axhline(FLOOR_UNI2H, ls="--", lw=1, color="#dd6b20", alpha=0.6, label=f"UNI2-h floor ({FLOOR_UNI2H})")
ax.set_xticks(x); ax.set_xticklabels(VARIANTS, rotation=20, ha="right")
ax.set_ylabel("QWK (5-fold CV mean ± std)"); ax.set_ylim(0.80, 0.97)
pass  # [TITLE_DISABLED S107] ax.set_title("CLAM-TR factorial: QWK by aggregator × encoder (bestHP)")
ax.legend(loc="lower left", framealpha=0.9)
plt.savefig(FIG_DIR / "fig_3_2_qwk_comparison.pdf"); plt.show()
print("S022 saved")

## CELL 2 — S023 encoder × aggregator interaction plot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
xs = [0, 1]
colors = sns.color_palette("Set2", len(VARIANTS))
for v, c in zip(VARIANTS, colors):
    y_uni, _   = get_bestHP(uni_exp, v)
    y_uni2h, _ = get_bestHP(uni2h_exp, v)
    ax.plot(xs, [y_uni, y_uni2h], marker="o", lw=2, color=c, label=v, markersize=8)
ax.set_xticks(xs); ax.set_xticklabels(["UNI (ViT-L/16)", "UNI2-h (ViT-H/14)"])
ax.set_ylabel("QWK (5-fold CV mean)")
pass  # [TITLE_DISABLED S107] ax.set_title("Encoder × aggregator interaction (4-vs-2 rank reversal)")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
ax.grid(True, alpha=0.4)
plt.savefig(FIG_DIR / "fig_3_3_encoder_aggregator_interaction.pdf"); plt.show()
print("S023 saved")

## CELL 3 — S024 per-grade accuracy heatmap (12 × 6)

In [ ]:
def per_grade(experiments_dict, variant):
    rec = experiments_dict.get(f"{variant}_bestHP", {})
    pg = rec.get("per_grade_acc_mean", {})
    # keys are str "0".."5"
    return [pg.get(str(i), np.nan) for i in range(6)]

rows, row_labels = [], []
for v in VARIANTS:
    rows.append(per_grade(uni_exp, v));   row_labels.append(f"UNI · {v}")
for v in VARIANTS:
    rows.append(per_grade(uni2h_exp, v)); row_labels.append(f"UNI2-h · {v}")

fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(np.array(rows), annot=True, fmt=".3f", cmap="YlOrRd", vmin=0.45, vmax=1.00,
            xticklabels=[f"G{i}" for i in range(6)], yticklabels=row_labels,
            cbar_kws={"label": "Per-grade accuracy"}, ax=ax, linewidths=0.4, linecolor="white")
ax.axhline(6, color="black", lw=2)
pass  # [TITLE_DISABLED S107] ax.set_title("Per-grade accuracy across 12 bestHP configurations")
plt.savefig(FIG_DIR / "fig_3_4_per_grade_heatmap.pdf"); plt.show()
print("S024 saved")

## CELL 4 — S025 confusion matrices (UNI RetNet + UNI2-h SpatialMultRetNet)

In [ ]:
def load_predictions(encoder_dir, variant_bestHP):
    """Aggregate y_true (labels) + y_pred (predictions) across 5 folds."""
    y_true, y_pred = [], []
    for fold in range(5):
        path = encoder_dir / "predictions" / f"{variant_bestHP}_fold{fold}_predictions.json"
        if not path.exists():
            print("missing:", path); continue
        with open(path) as f:
            data = json.load(f)
        y_true.extend(data["labels"])
        y_pred.extend(data["predictions"])
    return np.array(y_true), np.array(y_pred)

yt_uni, yp_uni = load_predictions(UNI_DIR,   "RetNet_bestHP")
yt_h,   yp_h   = load_predictions(UNI2H_DIR, "SpatialMultRetNet_bestHP")
print(f"UNI RetNet: {len(yt_uni)} samples; UNI2-h SpatialMultRetNet: {len(yt_h)} samples")

cm_uni = confusion_matrix(yt_uni, yp_uni, labels=range(6))
cm_h   = confusion_matrix(yt_h,   yp_h,   labels=range(6))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, cm, title in zip(axes, [cm_uni, cm_h], ["UNI · RetNet_bestHP", "UNI2-h · SpatialMultRetNet_bestHP"]):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=[f"G{i}" for i in range(6)],
                yticklabels=[f"G{i}" for i in range(6)], ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); pass  # [TITLE_DISABLED S107] ax.set_title(title)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_3_5_confusion_matrices.pdf"); plt.show()
print("S025 saved")

## CELL 5 — S027 + S013 BH-FDR waterfall (writes back q_BH + bonferroni)

In [ ]:
# This cell modifies statistical_tests.json by adding q_BH and bonferroni_passed
# fields to each comparison record. Closes both S013 (cross-check) and S027 (figure).

def load_pvals(encoder_dir):
    with open(encoder_dir / "statistical_tests.json") as f:
        data = json.load(f)  # list of dicts
    pvals = np.array([c["p_ttest"] for c in data])
    return data, pvals

for label, ed in [("UNI", UNI_DIR), ("UNI2-h", UNI2H_DIR)]:
    comps, pvals = load_pvals(ed)
    bh_pass, q_bh = fdrcorrection(pvals, alpha=0.05, method="indep")
    bf_pass, _, _, _ = multipletests(pvals, alpha=0.05, method="bonferroni")
    for c, q, bp in zip(comps, q_bh, bf_pass):
        c["q_BH"] = float(q)
        c["bonferroni_passed"] = bool(bp)
    with open(ed / "statistical_tests.json", "w") as f:
        json.dump(comps, f, indent=2)
    print(f"{label}: smallest q_BH = {min(q_bh):.4f}; written back ({len(comps)} comparisons).")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, label, ed in zip(axes, ["UNI", "UNI2-h"], [UNI_DIR, UNI2H_DIR]):
    with open(ed / "statistical_tests.json") as f:
        data = json.load(f)
    comps = sorted(data, key=lambda c: c["q_BH"])
    names = [c["comparison"] for c in comps]
    qs    = [c["q_BH"]       for c in comps]
    ds    = [c["cohens_d"]   for c in comps]
    cmap = plt.cm.RdYlGn_r; norm = plt.Normalize(vmin=0, vmax=2.0)
    ax.barh(range(len(comps)), qs, color=[cmap(norm(abs(d))) for d in ds], edgecolor="black")
    ax.axvline(0.05, ls="--", color="red", lw=1.5, label="q = 0.05 (BH threshold)")
    ax.set_yticks(range(len(comps))); ax.set_yticklabels(names, fontsize=8)
    ax.set_xlabel("BH-FDR q-value"); pass  # [TITLE_DISABLED S107] ax.set_title(f"{label} — pairwise comparisons")
    ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_3_7_bh_fdr_waterfall.pdf"); plt.show()
print("S027 + S013 saved")

## CELL 6 — S028 Cohen's d × q-value quadrants

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for label, ed, color in [("UNI", UNI_DIR, "#2b6cb0"), ("UNI2-h", UNI2H_DIR, "#dd6b20")]:
    with open(ed / "statistical_tests.json") as f:
        data = json.load(f)
    ds = [c["cohens_d"] for c in data]
    qs = [c["q_BH"]     for c in data]
    ax.scatter(np.abs(ds), qs, c=color, label=label, s=70, alpha=0.8, edgecolor="black")
ax.axhline(0.05, ls="--", color="red", lw=1, label="q = 0.05")
ax.axvline(0.8, ls=":", color="grey", lw=1, label="|d| = 0.8 (large)")
ax.set_xlabel("|Cohen's d|"); ax.set_ylabel("BH-FDR q-value")
ax.set_yscale("log")
pass  # [TITLE_DISABLED S107] ax.set_title("Statistical decision quadrants (sample-size-bounded power)")
ax.legend(); ax.grid(alpha=0.3)
plt.savefig(FIG_DIR / "fig_3_8_d_vs_q_quadrants.pdf"); plt.show()
print("S028 saved")

## CELL 7 — S029 external anchor comparison

In [ ]:
labels = [
    "UNI · RetNet_bestHP\n(this thesis)",
    "UNI2-h · SpatialMultRetNet_bestHP\n(this thesis)",
    "Kaggle PANDA 2020\n1st-place private LB",
    "UNI release\nFM benchmark range",
]
values = [0.9543, 0.9530, 0.9409, 0.9450]
errors = [0.0046, 0.0029, 0,      0.0050]
colors = ["#2b6cb0", "#dd6b20", "#718096", "#9f7aea"]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(labels, values, yerr=errors, capsize=4, color=colors, edgecolor="black")
ax.set_ylabel("QWK"); ax.set_ylim(0.93, 0.97)
pass  # [TITLE_DISABLED S107] ax.set_title("CLAM-TR vs external anchors (D23 split-caveat)")
ax.tick_params(axis="x", labelsize=8.5)
for i, v in enumerate(values):
    ax.text(i, v + 0.001, f"{v:.4f}", ha="center", fontsize=9)
plt.savefig(FIG_DIR / "fig_3_9_external_anchors.pdf"); plt.show()
print("S029 saved")

## CELL 8 — VERIFY all 7 PDFs landed

In [ ]:
expected = [
    "fig_3_2_qwk_comparison.pdf",
    "fig_3_3_encoder_aggregator_interaction.pdf",
    "fig_3_4_per_grade_heatmap.pdf",
    "fig_3_5_confusion_matrices.pdf",
    "fig_3_7_bh_fdr_waterfall.pdf",
    "fig_3_8_d_vs_q_quadrants.pdf",
    "fig_3_9_external_anchors.pdf",
]
for f in expected:
    p = FIG_DIR / f
    size_kb = p.stat().st_size // 1024 if p.exists() else 0
    print(f"{'✓' if p.exists() else '✗'}  {f}  ({size_kb} KB)")